# 📦 GGUF Export & Quantization — Where Is My Paisa

Converts a fine-tuned model to GGUF format for use with **llama.cpp** in the Android app.

Quantization variants produced:
- `Q4_K_M` — default production (best size/quality)
- `Q5_K_M` — premium quality
- `Q3_K_M` — extreme low-RAM fallback

Also generates the `manifest.json` required by the app's model catalog API.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
    "unsloth[colab-new]", "huggingface_hub", "--quiet", "--upgrade"], check=True)
print("✅ Ready")

In [ ]:
# @title Configuration
# Path to merged (float16) checkpoint — output of any finetune notebook's "merged_float16" cell
MERGED_MODEL_PATH = "outputs/llama_3.2_1b-finance/merged_float16"
MODEL_ID_BASE = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_SHORT = "llama-3.2-1b"
VERSION = "1.0.0"
OUTPUT_GGUF_DIR = f"gguf/{MODEL_SHORT}-finance"
import os
os.makedirs(OUTPUT_GGUF_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_GGUF_DIR}")

In [ ]:
# @title Load and export GGUF (all quantization levels)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MERGED_MODEL_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = False,  # load as float16 for export
)

# Export all quantization variants
quants = ["q4_k_m", "q5_k_m", "q3_k_m"]
for quant in quants:
    print(f"Exporting {quant.upper()}...")
    model.save_pretrained_gguf(OUTPUT_GGUF_DIR, tokenizer, quantization_method=quant)
    print(f"  ✅ {quant.upper()} done")

from pathlib import Path
print("\nGenerated files:")
for f in Path(OUTPUT_GGUF_DIR).glob("*.gguf"):
    print(f"  {f.name}: {f.stat().st_size / 1024**2:.1f} MB")

In [ ]:
# @title Validate GGUF files
from pathlib import Path

def validate_gguf(path):
    with open(path, "rb") as f:
        magic = f.read(4)
    valid = magic == b"GGUF"
    size_mb = Path(path).stat().st_size / 1024**2
    return valid, size_mb

print("GGUF validation:")
for f in Path(OUTPUT_GGUF_DIR).glob("*.gguf"):
    valid, size = validate_gguf(f)
    status = "✅ valid" if valid else "❌ INVALID"
    print(f"  {f.name}: {status} ({size:.1f} MB)")

In [ ]:
# @title Generate manifest.json per model catalog API spec
import hashlib, json, time
from pathlib import Path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

gguf_files = list(Path(OUTPUT_GGUF_DIR).glob("*.gguf"))
q4_file = next((f for f in gguf_files if "q4_k_m" in f.name.lower()), gguf_files[0])
q5_file = next((f for f in gguf_files if "q5_k_m" in f.name.lower()), None)

manifests = {}
for quant_label, quant_file in [("Q4_K_M", q4_file), ("Q5_K_M", q5_file)]:
    if quant_file is None:
        continue
    sha = sha256_file(quant_file)
    size_bytes = quant_file.stat().st_size
    min_ram = {"Q4_K_M": 6000, "Q5_K_M": 6000, "Q3_K_M": 4000}[quant_label]

    manifest = {
        "id": f"{MODEL_SHORT}-fin-{quant_label.lower().replace('_', '')}-v{VERSION}",
        "displayName": f"{MODEL_SHORT.replace('-', ' ').title()} Finance {quant_label}",
        "baseModel": MODEL_ID_BASE,
        "finetuneMethod": "QLoRA-SFT",
        "quantization": quant_label,
        "sizeBytes": size_bytes,
        "sha256": sha,
        "requirements": {
            "minRamMb": min_ram,
            "recommendedFreeStorageMb": int(size_bytes / 1024**2 * 1.5),
            "abis": ["arm64-v8a"],
            "minAndroidApi": 26,
        },
        "metrics": {
            "qualityScore": None,    # fill after running 07_eval.py
            "medianDecodeTokensPerSec": None,
            "medianFirstTokenMs": None,
            "hallucinationRisk": "medium",
            "batteryImpact": "medium",
        },
        "safety": {
            "refusalRecall": None,
            "hallucinatedNumberRate": None,
        },
        "provenance": {
            "datasetVersion": "wimp-finance-instruct-v1",
            "trainRunId": f"run_{time.strftime('%Y_%m_%d')}_{MODEL_SHORT}",
            "evalRunId": None,
        },
    }
    manifests[quant_label] = manifest
    out_path = f"{OUTPUT_GGUF_DIR}/manifest_{quant_label}.json"
    with open(out_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"✅ Manifest: {out_path}")
    print(f"   id: {manifest['id']}")
    print(f"   sha256: {sha[:16]}...")
    print(f"   size: {size_bytes / 1024**2:.1f} MB")

In [ ]:
# @title (Optional) Upload to HuggingFace Hub
# from huggingface_hub import login, HfApi
# login(token="hf_YOUR_TOKEN_HERE")
# api = HfApi()
# repo_id = f"YOUR_HF_USERNAME/wimp-{MODEL_SHORT}-finance"
# api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
# for f in Path(OUTPUT_GGUF_DIR).iterdir():
#     api.upload_file(path_or_fileobj=str(f), path_in_repo=f.name, repo_id=repo_id, repo_type="model")
# print(f"✅ Uploaded to {repo_id}")
print("HF upload skipped. Uncomment above to enable.")